In [1]:
import pandas as pd
import numpy as np

In [8]:
# Load database
df_database = pd.read_csv('../data/car_database.csv', sep=';')

In [9]:
# 1. Data Cleanup
if 'transmission' in df_database.columns:
    df_database['transmission'] = df_database['transmission'].apply(lambda x: 1 if str(x).lower().strip() == 'automatic' else 0)
if 'ground_clearance' in df_database.columns:
    df_database['ground_clearance'] = df_database['ground_clearance'].apply(lambda x: 2 if str(x).lower().strip() == 'high' else 1 if str(x).lower().strip() == 'medium' else 0)

# 2. Define Criteria and Optimization Direction
criteria_config = {
    'cost': 'min', 
    'horsepower': 'max', 
    'transmission': 'max', 
    'city_fuel_economy': 'max', 
    'ground_clearance': 'max', 
    'rear_power_windows': 'max', 
    'power_side_mirrors': 'max', 
    'infotainment_system': 'max', 
    'rear_parking_sensors': 'max', 
    'fog_lights': 'max', 
    'roof_rails': 'max'
}

# 3. Data Preprocessing
df_ahp = df_database.copy()

# Convert boolean/object columns to numeric values (0 or 1 for booleans)
# This ensures all criteria can be processed mathematically
for col in criteria_config.keys():
    # Convert boolean columns to integer (True=1, False=0)
    if df_ahp[col].dtype == 'bool':
        df_ahp[col] = df_ahp[col].astype(int)
    
    # Force conversion to numeric, coercing errors to NaN
    df_ahp[col] = pd.to_numeric(df_ahp[col], errors='coerce')

# Drop rows that have missing values (NaN) in the critical criteria columns
df_ahp = df_ahp.dropna(subset=criteria_config.keys())

# 4. Gaussian AHP Implementation (Gaussian Normalization)
# Initialize a dataframe to store normalized values
norm_matrix = pd.DataFrame(index=df_ahp.index)

# Step 4a: Normalization
# For 'max' criteria: x / sum(x)
# For 'min' criteria: (1/x) / sum(1/x)
for col, direction in criteria_config.items():
    if direction == 'max':
        norm_matrix[col] = df_ahp[col] / df_ahp[col].sum()
    else:
        # Invert values for minimization criteria (e.g., Cost)
        # Avoid division by zero if cost is 0 (unlikely for cars, but good practice)
        inv_values = 1 / df_ahp[col]
        norm_matrix[col] = inv_values / inv_values.sum()

# Step 4b: Calculate Gaussian Weights (Objective Weighting)
# Calculate the Mean and Standard Deviation for each normalized criterion
means = norm_matrix.mean()
stds = norm_matrix.std()

# Calculate the Gaussian Factor (Coefficient of Variation)
# Logic: Criteria with higher variability (high StdDev relative to Mean) 
# provide more information for differentiation and thus get higher weights.
gaussian_factors = stds / means

# Normalize the factors so that the sum of weights equals 1
weights = gaussian_factors / gaussian_factors.sum()

# Step 4c: Final Scoring
# Calculate the weighted sum for each alternative (car)
df_ahp['ahp_score'] = (norm_matrix * weights).sum(axis=1)

# 5. Ranking
# Rank the cars based on the score (Higher score is better)
df_ahp['rank'] = df_ahp['ahp_score'].rank(ascending=False)

# Sort the dataset by rank
final_results = df_ahp.sort_values('rank')

# 6. Display Results
print("Calculated Weights (Gaussian AHP):")
print(weights.sort_values(ascending=False))
print("\nTop 5 Ranked Cars:")
final_results[['car', 'version', 'cost', 'ahp_score']].head(5)

Calculated Weights (Gaussian AHP):
roof_rails              0.229128
fog_lights              0.174999
transmission            0.161197
rear_parking_sensors    0.137281
ground_clearance        0.121243
rear_power_windows      0.062361
power_side_mirrors      0.062361
horsepower              0.029669
cost                    0.014582
city_fuel_economy       0.007181
infotainment_system     0.000000
dtype: float64

Top 5 Ranked Cars:


,car,version,cost,ahp_score
30,Citroën C3 2026,1.0 TURBO 200 FLEX YOU CVT,110280.0,0.093407
7,Fiat Argo 2026,1.3 FIREFLY FLEX TREKKING CVT + Trekking TOP,113880.0,0.093127
2,Fiat Argo 2026,1.3 FIREFLY FLEX TREKKING MANUAL,102790.0,0.078517
3,Fiat Argo 2026,1.3 FIREFLY FLEX TREKKING MANUAL + Trekking TOP,105880.0,0.078499
6,Fiat Argo 2026,1.3 FIREFLY FLEX TREKKING CVT,110790.0,0.078474
